<a href="https://colab.research.google.com/github/nehemiahkalikenka/CSC4792_Group_30_Lusangazi_Town_Council_Data_Mining/blob/eda-data-quality-check/CSC_4792_ASSIGNMENT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lusangazi Town Council Multi-Source Dataset

CSC 4792: Data Mining and Warehousing

Group 30

This notebook documents the collection, extraction, cleaning,
preprocessing and preparation of data relating to Lusangazi
Town Council.

Step 1: Install & Import Dependencies
Sets up the execution environment, installs required scraping/parsing packages, and suppresses SSL verification warnings caused by misconfigured target server certificates.

In [ ]:
!pip install requests beautifulsoup4 pandas lxml pymupdf

In [ ]:
!pip install pdfplumber

Step 2: Site Crawling & Link Extraction
Crawls the official Lusangazi Town Council domain ([https://www.lusangazicouncil.gov.zm/](https://www.lusangazicouncil.gov.zm/)) to discover internal subpages and target document sections.

In [ ]:
import requests, os, re, time
import pandas as pd
import fitz
from bs4 import BeautifulSoup
from urllib.parse import urljoin, urlparse
import urllib3


urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

BASE_URL = "https://www.lusangazicouncil.gov.zm/"

def get_internal_links(base_url):

    response = requests.get(base_url, headers={"User-Agent": "Mozilla/5.0"}, timeout=30, verify=False)
    soup = BeautifulSoup(response.text, "html.parser")
    links = set()
    for a in soup.find_all("a", href=True):
        url = urljoin(base_url, a["href"])
        if urlparse(url).netloc == urlparse(base_url).netloc:
            links.add(url)
    return list(links)

internal_urls = get_internal_links(BASE_URL)
print(f"Discovered {len(internal_urls)} internal URLs.")

Step 3: PDF Document & Table Discovery
Scans target pages (including known CDF project and administrative pages) for embedded HTML tables and downloadable PDF links.

In [ ]:
import pandas as pd
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin, urlparse

# Additional direct targets known to host Lusangazi datasets/documents
target_urls = list(set(internal_urls + [
    "https://www.lusangazicouncil.gov.zm/?page_id=932",   # CDF Tracker / Projects
    "https://www.lusangazicouncil.gov.zm/?page_id=2884",  # CDF Main Page
    "https://www.lusangazicouncil.gov.zm/?page_id=118",   # About / Wards / Admin
]))

pdf_urls = set()
page_texts = []
extracted_tables = []

headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"}

print(f"Scanning {len(target_urls)} pages...")

for url in target_urls:
    try:
        resp = requests.get(url, headers=headers, timeout=15, verify=False)
        if resp.status_code != 200:
            continue

        soup = BeautifulSoup(resp.text, "html.parser")

        # 1. Broad PDF Discovery (scrapes <a> hrefs and direct string matches)
        for a in soup.find_all("a", href=True):
            href = a["href"].strip()
            if ".pdf" in href.lower():
                full_pdf_url = urljoin(url, href)
                pdf_urls.add(full_pdf_url)

        # 2. Extract HTML tables (handles non-standard HTML table structures)
        try:
            tables = pd.read_html(resp.text)
            for t in tables:
                if not t.empty:
                    extracted_tables.append((url, t))
        except Exception:
            pass

        # 3. Store raw page text for fallbacks
        text_content = soup.get_text(separator=" ", strip=True)
        if len(text_content) > 100:
            page_texts.append({"url": url, "text": text_content})

    except Exception as e:
        continue

pdf_urls = list(pdf_urls)

print(f"--- Scan Results ---")
print(f"Found {len(extracted_tables)} HTML tables")
print(f"Found {len(pdf_urls)} PDF documents")
print(f"Extracted content from {len(page_texts)} web pages")

# Preview discovered PDFs if found
if pdf_urls:
    print("\nDiscovered PDF URLs:")
    for p in pdf_urls[:10]:
        print(" -", p)

Step 4: Automated PDF Download & Content Parsing
Downloads all discovered PDF documents into a local directory (downloaded_pdfs/) and parses unstructured text content line-by-line using PyMuPDF.

In [ ]:
import fitz  # PyMuPDF
import requests
import os
import pandas as pd
import urllib3

urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

os.makedirs("downloaded_pdfs", exist_ok=True)
pdf_data = []

headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"}

print(f"Starting download and extraction for {len(pdf_urls)} PDFs...\n")

for idx, p_url in enumerate(pdf_urls, 1):
    try:
        filename = os.path.join("downloaded_pdfs", f"doc_{idx}.pdf")
        print(f"[{idx}/{len(pdf_urls)}] Downloading: {p_url}")

        # Download PDF file with SSL verification disabled
        r = requests.get(p_url, headers=headers, timeout=30, verify=False)
        with open(filename, "wb") as f:
            f.write(r.content)

        # Parse text and tables using PyMuPDF
        doc = fitz.open(filename)
        full_text = ""
        for page in doc:
            full_text += page.get_text() + "\n"

        pdf_data.append({
            "url": p_url,
            "filename": filename,
            "page_count": len(doc),
            "text": full_text
        })
        print(f"   └─ Successfully parsed {len(doc)} pages.")

    except Exception as e:
        print(f"   └─ Failed to download/parse: {e}")

print(f"\nSuccessfully downloaded and processed {len(pdf_data)} PDFs.")

### Step 5: Refactored Document Audit & Classification
We categorize each of the 72 downloaded PDF files by analyzing their text content for contextual keywords (CDF/Projects, Budgets/Financials, IDPs/Wards, or Policy/Admin) to structure a clean, audit-compliant document inventory catalog.

In [ ]:
import os
import pandas as pd

catalog_records = []

# Refined classification categories mapping keywords to labels
categories_map = {
    "Constituency Development Fund (CDF) & Community Projects": [
        "cdf", "constituency development", "bursary", "empowerment", "community projects", "contractor"
    ],
    "Annual Budgets & Financial Statements": [
        "budget", "financial statement", "revenue", "expenditure", "audit", "public finance", "lgef"
    ],
    "Integrated Development Plans & Strategic Priorities": [
        "idp", "integrated development", "strategic plan", "stakeholder engagement", "prioritised", "wdc"
    ],
    "Council Administration, Acts & Policies": [
        "act", "policy", "by-law", "regulation", "minutes", "standing orders", "local government"
    ]
}

for idx, item in enumerate(pdf_data, 1):
    raw_text = item.get("text", "").lower()
    url_lower = item.get("url", "").lower()
    text_to_scan = raw_text + " " + url_lower

    # Determine matching categories
    matched_cats = []
    for category, keywords in categories_map.items():
        if any(keyword in text_to_scan for keyword in keywords):
            matched_cats.append(category)

    # Fallback to general category if no keyword triggers
    category_label = matched_cats[0] if matched_cats else "General Council Policy & Admin"

    # Clean and extract metadata
    doc_url = item.get("url", "https://www.lusangazicouncil.gov.zm/")
    raw_filename = os.path.basename(item.get("filename", f"doc_{idx}.pdf"))

    # Clean and normalize document title for a professional catalog appearance
    clean_title = raw_filename.replace(".pdf", "").replace("-", " ").replace("_", " ").title()

    catalog_records.append({
        "document_id": f"DOC-LUS-{idx:03d}",
        "document_title": clean_title,
        "category": category_label,
        "year": 2024,
        "source_url": doc_url
    })

# Create a structured Pandas DataFrame
df_docs_catalog = pd.DataFrame(catalog_records)

# Clean columns to ensure uniform styling (lower case, snake_case)
df_docs_catalog.columns = df_docs_catalog.columns.astype(str).str.lower().str.strip().str.replace(" ", "_")

# Save pipe-delimited file adhering to naming specifications
output_catalog_path = "db-unza26-csc4792-lusangazi_council_documents.csv"
df_docs_catalog.to_csv(output_catalog_path, sep="|", index=False, encoding="utf-8")

print(f"[SUCCESS] Classified and saved {len(df_docs_catalog)} document records to '{output_catalog_path}'")
display(df_docs_catalog.head(15))

### Step 6: CDF Projects Data Extraction & Cleaning
This step extracts and compiles the list of Constituency Development Fund (CDF) and community projects by parsing discovered HTML tables and downloading PDFs. The script categorizes sectors (e.g., Water & Sanitation, Education, Roads), normalizes ward names, extracts numeric monetary values, and formats the output into a structured, audit-compliant table (`db-unza26-csc4792-lusangazi_cdf_projects.csv`). All image-only and non-project elements are automatically excluded.

In [ ]:
import os
import re
import pandas as pd
import pdfplumber

# Real target columns schema for File 1:
# project_id | project_name | ward | sector | amount | status | year

real_cdf_projects = []
project_counter = 1

# Helper function to categorize sectors correctly based on context keywords
def categorize_sector(text):
    t = str(text).lower()
    if any(k in t for k in ["road", "bridge", "culvert", "crossing", "feeder"]):
        return "Transport & Infrastructure"
    elif any(k in t for k in ["borehole", "water", "well", "wash", "piped"]):
        return "Water & Sanitation"
    elif any(k in t for k in ["school", "classroom", "desk", "teacher", "education", "ablution"]):
        return "Education"
    elif any(k in t for k in ["clinic", "health", "health post", "maternity", "ward"]):
        return "Health"
    elif any(k in t for k in ["bursary", "skills", "empowerment", "grant", "loan", "youth", "women"]):
        return "Social Empowerment & Skills"
    elif any(k in t for k in ["agriculture", "farming", "dip tank", "livestock"]):
        return "Agriculture & Livestock"
    else:
        return "Community Development"

# Helper function to clean Ward names
def clean_ward(ward_text):
    if not ward_text or pd.isna(ward_text):
        return "District-wide"
    w = str(ward_text).strip().title()
    w = re.sub(r'\s+Ward$', '', w, flags=re.IGNORECASE)
    # Check if we got index numbers or headers accidentally
    if w.isdigit() or len(w) < 3 or any(h in w.lower() for h in ["name", "group", "total", "serial", "ward"]):
        return "District-wide"
    return f"{w} Ward"

# Helper function to parse numeric monetary amounts from string representations safely
def parse_amount(val):
    if not val or pd.isna(val):
        return 0.0
    cleaned = re.sub(r'[^\d.]', '', str(val).replace(',', ''))
    try:
        return float(cleaned) if cleaned else 0.0
    except ValueError:
        return 0.0

# -------------------------------------------------------------------------
# PART 1: Process Discovered HTML Tables (Step 3 Output)
# -------------------------------------------------------------------------
print("Processing HTML Tables...")
for url, df_table in extracted_tables:
    df_table = df_table.dropna(how='all')
    # Scan tables that appear to contain project information
    header_str = " ".join(df_table.columns.astype(str)).lower()

    if any(k in header_str for k in ["project", "ward", "cost", "amount", "bursary", "empowerment"]):
        for _, row in df_table.iterrows():
            row_vals = [str(x).strip() for x in row.values if pd.notna(x)]
            row_str = " ".join(row_vals).lower()

            # Skip header rows
            if "project name" in row_str or "serial" in row_str or len(row_vals) < 2:
                continue

            # Parse project details
            proj_name = row_vals[1] if len(row_vals) > 1 else row_vals[0]
            if len(proj_name) < 5 or proj_name.isdigit():
                continue

            ward_val = "District-wide"
            amount_val = 0.0
            status_val = "Approved"

            # Attempt to extract ward and cost dynamically
            for val in row_vals:
                # Look for ward indicators
                if "ward" in str(val).lower() and len(str(val)) > 4:
                    ward_val = clean_ward(val)
                # Look for potential cost values
                if re.match(r'^\d{1,3}(,\d{3})+(\.\d{2})?$', str(val)) or (str(val).isdigit() and float(val) > 1000):
                    amount_val = parse_amount(val)

            real_cdf_projects.append({
                "project_id": f"LUS-CDF-{project_counter:03d}",
                "project_name": proj_name.strip(),
                "ward": ward_val,
                "sector": categorize_sector(proj_name),
                "amount": amount_val,
                "status": "Completed" if "completed" in row_str else "Ongoing" if "ongoing" in row_str else "Approved",
                "year": 2024
            })
            project_counter += 1

# -------------------------------------------------------------------------
# PART 2: Process Downloaded PDF Documents (Step 4 Output)
# -------------------------------------------------------------------------
print("Processing PDF Documents...")
for item in pdf_data:
    filename = item.get("filename", "")
    raw_text = item.get("text", "")

    # Target files related to projects, approvals, and CDF allocations
    if any(k in filename.lower() or k in raw_text.lower()[:300].lower() for k in ["project", "cdf", "bursary", "empowerment"]):
        try:
            with pdfplumber.open(filename) as pdf:
                for page in pdf.pages[:5]:  # Process the first few pages of each file to avoid performance bottlenecks
                    tables = page.extract_tables()
                    if not tables:
                        continue
                    for tbl in tables:
                        for r_idx, row in enumerate(tbl):
                            clean_row = [str(cell).strip() for cell in row if cell is not None]
                            row_str = " ".join(clean_row).lower()

                            if r_idx == 0 or len(clean_row) < 3 or "project" in row_str or "serial" in row_str:
                                continue

                            proj_name = clean_row[1] if len(clean_row) > 1 else clean_row[0]
                            if len(proj_name) < 6 or proj_name.isdigit():
                                continue

                            # Extract ward
                            ward_candidate = "District-wide"
                            for cell in clean_row:
                                if any(w in cell.lower() for w in ["ward", "central", "ukwimi", "nyakawise", "chikowa"]):
                                    ward_candidate = clean_ward(cell)
                                    break

                            # Parse amount
                            amount_candidate = 0.0
                            for cell in clean_row:
                                if re.search(r'\d{3,}', cell.replace(",", "")):
                                    parsed = parse_amount(cell)
                                    if parsed > 500:  # Valid monetary thresholds
                                        amount_candidate = parsed
                                        break

                            real_cdf_projects.append({
                                "project_id": f"LUS-CDF-{project_counter:03d}",
                                "project_name": proj_name.strip(),
                                "ward": ward_candidate,
                                "sector": categorize_sector(proj_name),
                                "amount": amount_candidate,
                                "status": "Completed" if "completed" in row_str else "Ongoing" if "ongoing" in row_str else "Approved",
                                "year": 2024
                            })
                            project_counter += 1
        except Exception as e:
            continue

# -------------------------------------------------------------------------
# PART 3: Create DataFrame, Clean and Save Output
# -------------------------------------------------------------------------
if real_cdf_projects:
    df_projects_clean = pd.DataFrame(real_cdf_projects)
else:
    # Fallback to create schema structure
    df_projects_clean = pd.DataFrame(columns=["project_id", "project_name", "ward", "sector", "amount", "status", "year"])

# Drop exact duplicates
df_projects_clean.drop_duplicates(subset=["project_name", "ward", "amount"], inplace=True)

# Ensure output directories or final structures match pipe format
output_csv_path = "db-unza26-csc4792-lusangazi_cdf_projects.csv"
df_projects_clean.to_csv(output_csv_path, sep="|", index=False, encoding="utf-8")

# Save a duplicate to exported_csvs folder to prevent breaking EDA scripts
os.makedirs("exported_csvs", exist_ok=True)
df_projects_clean.to_csv(os.path.join("exported_csvs", output_csv_path), sep="|", index=False, encoding="utf-8")

print(f"\n[SUCCESS] Successfully parsed, cleaned, and exported {len(df_projects_clean)} actual CDF records to '{output_csv_path}'.")
display(df_projects_clean.head(15))

### Step 7: extracting wards_wdc.csv

In [ ]:
import os
import re
import pandas as pd
import pdfplumber
import requests
from bs4 import BeautifulSoup

# File output name matching assignment naming rules
OUTPUT_CSV = "db-unza26-csc4792-lusangazi_financials_budget.csv"

# Keywords targeting financial footprints
FINANCIAL_KEYWORDS = [
    "budget", "lgef", "revenue", "expenditure", "equalisation",
    "grant", "levy", "fees", "tax", "allocation", "zmw", "kwa"
]

records = []

# Helper to clean strings
def clean_str(val):
    return re.sub(r'\s+', ' ', str(val)).strip() if val else ""

# ---------------------------------------------------------
# PART 1: Parse HTML tables directly from target_urls
# ---------------------------------------------------------
print("--- Extracting Financial Data from Web Pages ---")
headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"}

# Use target_urls defined in your Step 3 cell
urls_to_scan = target_urls if 'target_urls' in locals() else [
    "https://www.lusangazicouncil.gov.zm/?page_id=932",
    "https://www.lusangazicouncil.gov.zm/?page_id=2884",
    "https://www.lusangazicouncil.gov.zm/?page_id=118"
]

for url in urls_to_scan:
    try:
        resp = requests.get(url, headers=headers, timeout=15, verify=False)
        if resp.status_code == 200:
            soup = BeautifulSoup(resp.content, "html.parser")
            tables = soup.find_all("table")

            for table_idx, table in enumerate(tables):
                rows = table.find_all("tr")
                for row in rows:
                    cols = [clean_str(ele.text) for ele in row.find_all(["td", "th"]) if clean_str(ele.text)]
                    if len(cols) >= 2:
                        line_str = " | ".join(cols)
                        if any(kw in line_str.lower() for kw in FINANCIAL_KEYWORDS):
                            amounts = re.findall(r'(?:ZMW\s*)?\(?\d{1,3}(?:,\d{3})*(?:\.\d{2})?\)?', line_str)
                            records.append({
                                "council_name": "Lusangazi Town Council",
                                "financial_year": "2024",
                                "category": cols[0],
                                "revenue_or_expenditure": "Revenue" if any(k in line_str.lower() for k in ["revenue", "grant", "levy", "tax", "lgef"]) else "Expenditure",
                                "item_description": cols[1] if len(cols) > 1 else cols[0],
                                "approved_budget_zmw": amounts[0] if len(amounts) > 0 else "N/A",
                                "actual_collected_spent_zmw": amounts[1] if len(amounts) > 1 else "N/A",
                                "source_document": url,
                                "source_type": "HTML Table"
                            })
    except Exception as e:
        print(f"Skipping {url} due to error: {e}")

# ---------------------------------------------------------
# PART 2: Extract Financial Tables & Text from PDF Files
# ---------------------------------------------------------
print("\n--- Extracting Financial Data from PDF Files ---")
PDF_DIR = "./pdfs"  # Adjust if saved elsewhere (e.g., './downloaded_pdfs' or '/content/')

if os.path.exists(PDF_DIR):
    pdf_files = [f for f in os.listdir(PDF_DIR) if f.lower().endswith('.pdf')]
    for pdf_file in pdf_files:
        pdf_path = os.path.join(PDF_DIR, pdf_file)
        try:
            with pdfplumber.open(pdf_path) as pdf:
                for page_idx, page in enumerate(pdf.pages, start=1):
                    text = page.extract_text() or ""

                    if any(kw in text.lower() for kw in FINANCIAL_KEYWORDS):
                        tables = page.extract_tables()

                        # Process tables in PDF
                        for table in tables:
                            for row in table:
                                clean_row = [clean_str(cell) for cell in row if cell and clean_str(cell) != ""]
                                if len(clean_row) >= 2:
                                    line_str = " | ".join(clean_row)
                                    amounts = re.findall(r'(?:ZMW\s*)?\(?\d{1,3}(?:,\d{3})*(?:\.\d{2})?\)?', line_str)
                                    records.append({
                                        "council_name": "Lusangazi Town Council",
                                        "financial_year": "2024",
                                        "category": clean_row[0],
                                        "revenue_or_expenditure": "Revenue" if any(k in line_str.lower() for k in ["revenue", "grant", "lgef", "rates"]) else "Expenditure",
                                        "item_description": clean_row[1] if len(clean_row) > 1 else clean_row[0],
                                        "approved_budget_zmw": amounts[0] if len(amounts) > 0 else "N/A",
                                        "actual_collected_spent_zmw": amounts[1] if len(amounts) > 1 else "N/A",
                                        "source_document": pdf_file,
                                        "source_type": f"PDF Page {page_idx}"
                                    })
        except Exception as e:
            print(f"Could not read {pdf_file}: {e}")

# ---------------------------------------------------------
# PART 3: Save pipe-delimited CSV
# ---------------------------------------------------------
df_financials = pd.DataFrame(records)

if not df_financials.empty:
    df_financials.drop_duplicates(inplace=True)
    df_financials.to_csv(OUTPUT_CSV, sep='|', index=False, encoding='utf-8')
    print(f"\n[SUCCESS] Generated '{OUTPUT_CSV}' with {len(df_financials)} financial records.")
    display(df_financials.head())
else:
    print("\n[WARNING] No financial matching records were extracted. Verify PDF directory or URL responses.")

step 8: scraping HTML tables and PDF documents to generate the db-unza26-csc4792-lusangazi_idp_priorities.csv by targeting key sector priorities, strategic objectives, target wards, costs, and implementation timelines.

In [ ]:
import os
import re
import pandas as pd
import pdfplumber
import requests
from bs4 import BeautifulSoup

# Output filename matching the schema specifications
OUTPUT_CSV = "db-unza26-csc4792-lusangazi_idp_priorities.csv"

# Keywords targeting Integrated Development Plan (IDP) strategic projects & priorities
IDP_KEYWORDS = [
    "idp", "integrated development plan", "priority", "strategic objective",
    "sector", "agriculture", "roads", "health", "education", "water", "sanitation"
]

records = []

def clean_str(val):
    return re.sub(r'\s+', ' ', str(val)).strip() if val else ""

# Helper to categorize development sectors based on line context
def infer_sector(text):
    text_lower = text.lower()
    if any(k in text_lower for k in ["road", "bridge", "infrastructure"]):
        return "Infrastructure & Roads"
    elif any(k in text_lower for k in ["health", "clinic", "hospital"]):
        return "Health"
    elif any(k in text_lower for k in ["school", "education", "classroom"]):
        return "Education"
    elif any(k in text_lower for k in ["water", "sanitation", "borehole", "wash"]):
        return "Water & Sanitation"
    elif any(k in text_lower for k in ["agri", "farm", "livestock"]):
        return "Agriculture"
    else:
        return "General Community Development"

# ---------------------------------------------------------
# PART 1: Scrape IDP Priorities from Web Pages (HTML)
# ---------------------------------------------------------
print("--- Extracting IDP Priorities from Web Pages ---")
headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"}

urls_to_scan = target_urls if 'target_urls' in locals() else [
    "https://www.lusangazicouncil.gov.zm/?page_id=932",
    "https://www.lusangazicouncil.gov.zm/?page_id=2884",
    "https://www.lusangazicouncil.gov.zm/?page_id=118"
]

idp_counter = 1

for url in urls_to_scan:
    try:
        resp = requests.get(url, headers=headers, timeout=15, verify=False)
        if resp.status_code == 200:
            soup = BeautifulSoup(resp.content, "html.parser")
            tables = soup.find_all("table")

            for table in tables:
                rows = table.find_all("tr")
                for row in rows:
                    cols = [clean_str(ele.text) for ele in row.find_all(["td", "th"]) if clean_str(ele.text)]
                    if len(cols) >= 2:
                        line_str = " | ".join(cols)
                        if any(kw in line_str.lower() for kw in IDP_KEYWORDS):
                            amounts = re.findall(r'(?:ZMW\s*)?\(?\d{1,3}(?:,\d{3})*(?:\.\d{2})?\)?', line_str)
                            years = re.findall(r'202[0-9](?:-202[0-9])?', line_str)

                            records.append({
                                "idp_id": f"IDP-LUS-{idp_counter:03d}",
                                "sector": infer_sector(line_str),
                                "strategic_objective": cols[0],
                                "target_ward": cols[1] if len(cols) > 2 else "District-wide",
                                "estimated_cost_zmw": amounts[0] if len(amounts) > 0 else "N/A",
                                "implementation_period": years[0] if len(years) > 0 else "2024-2026"
                            })
                            idp_counter += 1
    except Exception as e:
        print(f"Skipping {url} due to error: {e}")

# ---------------------------------------------------------
# PART 2: Extract IDP Priorities from PDF Documents
# ---------------------------------------------------------
print("\n--- Extracting IDP Priorities from PDF Files ---")
PDF_DIR = "./pdfs"

if os.path.exists(PDF_DIR):
    pdf_files = [f for f in os.listdir(PDF_DIR) if f.lower().endswith('.pdf')]
    for pdf_file in pdf_files:
        pdf_path = os.path.join(PDF_DIR, pdf_file)
        try:
            with pdfplumber.open(pdf_path) as pdf:
                for page_idx, page in enumerate(pdf.pages, start=1):
                    text = page.extract_text() or ""

                    if any(kw in text.lower() for kw in IDP_KEYWORDS):
                        tables = page.extract_tables()
                        for table in tables:
                            for row in table:
                                clean_row = [clean_str(cell) for cell in row if cell and clean_str(cell) != ""]
                                if len(clean_row) >= 2:
                                    line_str = " | ".join(clean_row)
                                    amounts = re.findall(r'(?:ZMW\s*)?\(?\d{1,3}(?:,\d{3})*(?:\.\d{2})?\)?', line_str)
                                    years = re.findall(r'202[0-9](?:-202[0-9])?', line_str)

                                    records.append({
                                        "idp_id": f"IDP-LUS-{idp_counter:03d}",
                                        "sector": infer_sector(line_str),
                                        "strategic_objective": clean_row[0],
                                        "target_ward": clean_row[1] if len(clean_row) > 2 else "District-wide",
                                        "estimated_cost_zmw": amounts[0] if len(amounts) > 0 else "N/A",
                                        "implementation_period": years[0] if len(years) > 0 else "2024-2026"
                                    })
                                    idp_counter += 1
        except Exception as e:
            print(f"Could not read {pdf_file}: {e}")

# ---------------------------------------------------------
# PART 3: Save Pipe-Delimited CSV
# ---------------------------------------------------------
df_idp = pd.DataFrame(records)

if not df_idp.empty:
    df_idp.drop_duplicates(subset=["sector", "strategic_objective", "target_ward"], inplace=True)
    df_idp.to_csv(OUTPUT_CSV, sep='|', index=False, encoding='utf-8')
    print(f"\n[SUCCESS] Generated '{OUTPUT_CSV}' with {len(df_idp)} IDP strategic priorities.")
    display(df_idp.head())
else:
    print("\n[WARNING] No IDP priorities matched the keywords. Verify source PDFs or target URLs.")

##Exploratory Data Analysis (EDA)

This section performs a basic Exploratory Data Analysis (EDA) on the CSV files generated in the previous step. The primary goals of this EDA are:

1.  **Inspect DataFrames**: Load each generated CSV file (`lusangazi_cdf_projects.csv`, `lusangazi_wards_wdc.csv`, `lusangazi_council_documents.csv`, `lusangazi_administration.csv`) into pandas DataFrames.
2.  **Understand Structure**: Examine the shape (number of rows and columns) and data types of each DataFrame.
3.  **Preview Data**: Display the first few rows of each DataFrame to get a glimpse of the data content.
4.  **Value Distribution**: Analyze the distribution of categorical variables using `value_counts()` and summarize numerical columns using `describe()` to identify potential issues or insights.

In [ ]:
import pandas as pd

df = pd.read_csv('/content/exported_csvs/db-unza26-csc4792-lusangazi_cdf_projects.csv', sep='|')
print('CDF PROJECTS:', df.shape)
print(df.dtypes)

In [ ]:
print(df.head(20).to_string())

In [ ]:
print(df['ward'].value_counts())
print(df['status'].value_counts())
print(df['year'].value_counts())
print(df['sector'].value_counts())
print(df['amount'].describe())

In [ ]:
df_wards = pd.read_csv('/content/exported_csvs/db-unza26-csc4792-lusangazi_wards_wdc.csv', sep='|')
print(df_wards['wdc_representative'].value_counts())
print(df_wards['zone'].value_counts())
print(df_wards['key_priorities'].value_counts())
print(df_wards.head(15).to_string())

df_docs = pd.read_csv('/content/exported_csvs/db-unza26-csc4792-lusangazi_council_documents.csv', sep='|')
print(df_docs['category'].value_counts())
print(df_docs.head(10).to_string())   # this one checked out clean

df_admin = pd.read_csv('/content/exported_csvs/db-unza26-csc4792-lusangazi_administration.csv', sep='|')
print(df_admin['key_functions'].nunique(), df_admin['official_contact'].nunique())
print(df_admin.head(10).to_string())